# Laboratorio 8 — Máquinas de Vectores de Soporte (SVM)
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 21 de octubre de 2026 · Clase 8
**Bloques:** 6-8 (11:40-13:40) · Lab. Central 114

Este laboratorio acompaña la clase teórica de SVM. La idea es *ver con los ojos* qué hacen el margen, el parámetro `C`, el parámetro `gamma` y el kernel, y cerrar con un flujo reproducible (Pipeline + validación cruzada + búsqueda de hiperparámetros).

## Objetivos de aprendizaje

1. Explicar qué es el margen y por qué maximizarlo mejora la generalización.
2. Interpretar el parámetro `C` (margen blando) y los parámetros del kernel RBF (`gamma`).
3. Explicar por qué un kernel permite fronteras no lineales sin calcular el espacio ampliado.
4. Entrenar una SVM con scikit-learn dentro de un `Pipeline` con escalado, y afinar `C` y `gamma`.

## Plan de la sesión (≈ 100 min)

| Parte | Tema | Tiempo sugerido |
|---|---|---|
| A | Margen y vectores de soporte | 15 min |
| B | Efecto de `C` | 15 min |
| C | Lineal vs RBF (y polinomial) | 15 min |
| D | `gamma` y `C` en una grilla | 20 min |
| E | Datos reales con escalado | 20 min |
| F | Afinar `C` y `gamma` | 10 min |
| G | Comparación y cierre | 5 min |

> **Cómo trabajar:** las celdas marcadas con `# TODO` o con `...` hay que completarlas. Después de cada parte hay una celda de **autocomprobación**: si falla, revisa tu código antes de seguir. Las preguntas se responden en las celdas de texto «**Respuesta**».

## Configuración

In [ ]:
# Configuración del entorno
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q scikit-learn matplotlib numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.datasets import make_blobs, make_moons, make_circles, load_breast_cancer
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 42
np.random.seed(SEED)

# Paleta del curso
MARINO, ROSA, GRIS = "#1b2740", "#c8807a", "#8b929c"
CMAP_PUNTOS = ListedColormap([MARINO, ROSA])
CMAP_FONDO = ListedColormap(["#c8d1de", "#f4e6d3"])

plt.rcParams.update({
    "figure.facecolor": "#f2f1ee", "axes.facecolor": "#fffefb",
    "axes.edgecolor": "#dcdad3", "axes.titleweight": "bold",
    "axes.titlecolor": MARINO, "axes.labelcolor": "#333b48",
    "axes.grid": True, "grid.color": "#e3e1da", "grid.linewidth": 0.8,
    "font.size": 11,
})

### Utilidad de graficado (ya está lista, no hay que modificarla)

`graficar_frontera` dibuja los datos, la frontera de decisión y, si `margen=True`, también las rectas de margen (`f(x) = ±1`) y los **vectores de soporte** (círculos vacíos).

In [ ]:
def graficar_frontera(modelo, X, y, ax=None, titulo="", margen=False):
    """Dibuja datos + frontera de decisión de un clasificador de 2 variables.

    margen=True  -> contornos de f(x) = -1, 0, +1 y vectores de soporte (solo para SVC).
    margen=False -> regiones de predicción coloreadas.
    """
    ax = ax or plt.gca()
    if margen:
        DecisionBoundaryDisplay.from_estimator(
            modelo, X, response_method="decision_function", plot_method="contour",
            levels=[-1, 0, 1], colors=[GRIS, MARINO, GRIS],
            linestyles=["--", "-", "--"], ax=ax)
        sv = modelo.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=170, facecolors="none",
                   edgecolors=ROSA, linewidths=1.8, label="vectores de soporte")
    else:
        DecisionBoundaryDisplay.from_estimator(
            modelo, X, response_method="predict", cmap=CMAP_FONDO,
            alpha=0.9, ax=ax, grid_resolution=250)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=CMAP_PUNTOS, s=28,
               edgecolors="white", linewidths=0.5)
    ax.set_title(titulo)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    return ax

---
## Parte A — Margen y vectores de soporte (15 min)

**Qué hacer**

1. Los datos son dos nubes de puntos: `make_blobs(n_samples=100, centers=2, cluster_std=1.0, random_state=3)` (la semilla se eligió para que las nubes sean linealmente separables).
2. Entrena `SVC(kernel="linear", C=1e3)` (un `C` muy grande penaliza casi sin tolerancia que un punto invada el margen: se parece a un margen duro).
3. Dibuja la frontera, los márgenes y los vectores de soporte con `graficar_frontera(..., margen=True)`.
4. Mira cuántos vectores de soporte hay (`modelo.n_support_` entrega uno por clase).
5. Elimina un punto que **no** es de soporte, reentrena y compara los coeficientes (`coef_`, `intercept_`). Repite eliminando un vector de soporte.

> **Nota técnica:** el optimizador de scikit-learn se detiene con una tolerancia (`tol=1e-3` por defecto), así que dos entrenamientos casi idénticos pueden diferir en el tercer decimal. Para comparar con rigor, el experimento del punto 5 usa `tol=1e-8`.

In [ ]:
X_a, y_a = make_blobs(n_samples=100, centers=2, cluster_std=1.0, random_state=3)

# 1) TODO: entrena una SVM lineal con C=1e3 y guárdala en `modelo_a`
modelo_a = ...

# 2) Frontera, márgenes y vectores de soporte (la utilidad ya está lista)
fig, ax = plt.subplots(figsize=(6.5, 4.8))
graficar_frontera(modelo_a, X_a, y_a, ax, "SVM lineal, C = 1e3", margen=True)
ax.legend(loc="best")
plt.show()

# TODO: imprime el nº de vectores de soporte por clase (pista: atributo n_support_),
# el total y el ancho del margen 2/||w|| (pista: modelo_a.coef_ y np.linalg.norm)
# TODO (opcional): ¿cuántos vectores de soporte tienen |alfa| = C? (pista: modelo_a.dual_coef_)
...

In [ ]:
# Experimento: ¿qué pasa si se quita un punto que NO es de soporte? ¿y uno que SÍ lo es?
idx_soporte = modelo_a.support_                       # índices de los vectores de soporte
idx_no_soporte = np.setdiff1d(np.arange(len(X_a)), idx_soporte)

def reentrenar_sin(indice=None):
    """Entrena SVC(kernel="linear", C=1e3, tol=1e-8) sin el punto `indice` (o con todos si es None)."""
    # TODO: arma una máscara booleana que deje fuera `indice` (si no es None),
    # entrena SVC(kernel="linear", C=1e3, tol=1e-8) con el resto y devuélvelo
    ...

referencia = reentrenar_sin()                          # mismo modelo, con tolerancia estricta
sin_no_soporte = reentrenar_sin(idx_no_soporte[0])
sin_soporte = reentrenar_sin(idx_soporte[0])

# TODO: compara coef_ e intercept_ de `referencia` con cada modelo reentrenado
# (pista: np.allclose con atol=1e-4) e imprime si la frontera es la misma
...

In [ ]:
# Autocomprobación — Parte A
assert isinstance(modelo_a, SVC), "Entrena modelo_a con SVC"
assert modelo_a.kernel == "linear" and modelo_a.C == 1e3, "Usa kernel='linear' y C=1e3"
assert hasattr(modelo_a, "support_vectors_"), "modelo_a todavía no está entrenado (.fit)"
assert sin_soporte is not None and sin_no_soporte is not None and referencia is not None, "reentrenar_sin debe devolver un modelo"
print("✔ Parte A: modelo entrenado y experimento de reentrenamiento completo")

**Pregunta A1:** ¿cuántos vectores de soporte hay? ¿Qué fracción de los datos representan?

**Respuesta:** (escribe aquí)

**Pregunta A2:** si se elimina un punto que *no* es de soporte y se reentrena, ¿cambia la frontera? ¿Y si se elimina un vector de soporte? ¿Qué dice eso sobre qué datos «mandan» en una SVM?

**Respuesta:** (escribe aquí)

---
## Parte B — Efecto de `C` (15 min)

**Qué hacer**

1. Genera nubes con traslape: `make_blobs(centers=2, cluster_std=2.5, random_state=8)` (la semilla se eligió para que el traslape sea visible).
2. Entrena `SVC(kernel="linear")` con `C ∈ {0.01, 1, 100}`.
3. Grafica los tres márgenes lado a lado y arma una tabla con: `C`, nº de vectores de soporte, ancho del margen `2/‖w‖` y exactitud de entrenamiento.

In [ ]:
X_b, y_b = make_blobs(centers=2, cluster_std=2.5, random_state=8)
valores_C = [0.01, 1, 100]

filas = []
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, C in zip(axes, valores_C):
    # TODO: entrena SVC(kernel="linear", C=C) con (X_b, y_b) y guárdalo en `m`
    m = ...
    # TODO: agrega a `filas` un dict con las claves "C", "n_vectores_soporte",
    # "ancho_margen" (2/||w||) y "exactitud_train" (m.score)
    filas.append({...})
    graficar_frontera(m, X_b, y_b, ax, f"C = {C}", margen=True)
plt.tight_layout()
plt.show()

tabla_b = pd.DataFrame(filas)
tabla_b

In [ ]:
# Autocomprobación — Parte B
assert len(tabla_b) == 3, "La tabla debe tener una fila por valor de C"
assert {"C", "n_vectores_soporte", "ancho_margen", "exactitud_train"} <= set(tabla_b.columns), \
    "Usa las claves C, n_vectores_soporte, ancho_margen y exactitud_train"
assert tabla_b["exactitud_train"].between(0.5, 1.0).all(), "La exactitud debe estar entre 0.5 y 1"
assert tabla_b["ancho_margen"].is_monotonic_decreasing, "El margen debería estrecharse al subir C (revisa 2/||w||)"
print("✔ Parte B: tabla completa")

**Pregunta B1:** ¿qué pasa con el margen y con el número de vectores de soporte al subir `C`? ¿Con cuál `C` el modelo es más «tolerante» a los errores?

**Respuesta:** (escribe aquí)

---
## Parte C — Lineal vs RBF (15 min)

**Qué hacer**

1. Usa `make_moons(noise=0.2)` y `make_circles(factor=0.4, noise=0.1)` (200 puntos cada uno, `random_state=SEED`).
2. Compara `kernel="linear"`, `"poly"` (`degree=3`, `coef0=1`) y `"rbf"` en cada conjunto y dibuja las fronteras con `DecisionBoundaryDisplay` (a través de la utilidad `graficar_frontera`, con `margen=False`).
3. Guarda la exactitud en un conjunto de prueba (30 %, estratificado) en la tabla `tabla_c`.

> **Nota sobre `coef0`:** con el valor por defecto `coef0=0`, el kernel polinomial $(\gamma\,\langle x, x'\rangle)^3$ sólo contiene términos de grado exactamente 3 y no puede representar $x_1^2 + x_2^2$ (la forma de un círculo). Con `coef0=1`, $(\gamma\,\langle x, x'\rangle + 1)^3$ incluye todos los grados hasta 3.

In [ ]:
conjuntos_c = {
    "lunas": make_moons(n_samples=200, noise=0.2, random_state=SEED),
    "círculos": make_circles(n_samples=200, factor=0.4, noise=0.1, random_state=SEED),
}
kernels = {
    "lineal": dict(kernel="linear"),
    "polinomial (grado 3)": dict(kernel="poly", degree=3, coef0=1),
    "RBF": dict(kernel="rbf"),
}

filas = []
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for fila, (nombre_datos, (X, y)) in enumerate(conjuntos_c.items()):
    # TODO: separa entrenamiento/prueba (test_size=0.3, stratify=y, random_state=SEED)
    X_tr, X_te, y_tr, y_te = ...
    for col, (nombre_k, params) in enumerate(kernels.items()):
        # TODO: entrena SVC(C=1, **params) con el conjunto de entrenamiento
        m = ...
        # TODO: calcula la exactitud en prueba y guarda una fila en `filas`
        acc = ...
        filas.append({"datos": nombre_datos, "kernel": nombre_k, "exactitud_test": round(acc, 3)})
        graficar_frontera(m, X, y, axes[fila, col], f"{nombre_datos} · {nombre_k} (test {acc:.2f})")
plt.tight_layout()
plt.show()

tabla_c = (pd.DataFrame(filas).pivot(index="datos", columns="kernel", values="exactitud_test")
           .reindex(columns=list(kernels)))   # mismo orden que en `kernels`
tabla_c

In [ ]:
# Autocomprobación — Parte C
assert tabla_c.shape == (2, 3), "tabla_c debe ser 2 conjuntos x 3 kernels"
assert ((tabla_c >= 0.4) & (tabla_c <= 1.0)).all().all(), "Exactitudes fuera de rango"
print("✔ Parte C: seis modelos entrenados y comparados")

**Pregunta C1:** ¿qué kernel usarían en cada conjunto (lunas y círculos) y por qué? ¿Qué le pasa al kernel lineal en los círculos concéntricos?

**Respuesta:** (escribe aquí)

---
## Parte D — `gamma` y `C` en una grilla (20 min)

**Qué hacer**

1. Con `make_moons(n_samples=300, noise=0.3)`, separa entrenamiento (70 %) y prueba (30 %, estratificado).
2. Entrena las 9 combinaciones de `C ∈ {0.1, 1, 10}` y `gamma ∈ {0.1, 1, 100}` con kernel RBF y dibújalas en una grilla de subgráficos 3×3 (filas = `C`, columnas = `gamma`).
3. Guarda exactitud de entrenamiento y de prueba de cada una en `tabla_d` y calcula la brecha (train − test).
4. Señala **a mano** (en la respuesta) cuál combinación subajusta y cuál sobreajusta.

In [ ]:
X_d, y_d = make_moons(n_samples=300, noise=0.3, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(X_d, y_d, test_size=0.3, stratify=y_d, random_state=SEED)

valores_C = [0.1, 1, 10]
valores_gamma = [0.1, 1, 100]

filas = []
fig, axes = plt.subplots(3, 3, figsize=(14, 13))
for i, C in enumerate(valores_C):
    for j, g in enumerate(valores_gamma):
        # TODO: entrena SVC(kernel="rbf", C=C, gamma=g) con el conjunto de entrenamiento
        m = ...
        # TODO: exactitud en entrenamiento (tr) y en prueba (te)
        tr, te = ..., ...
        filas.append({"C": C, "gamma": g, "train": round(tr, 3), "test": round(te, 3),
                      "brecha": round(tr - te, 3)})
        graficar_frontera(m, X_tr, y_tr, axes[i, j], f"C={C}, γ={g}\ntrain {tr:.2f} · test {te:.2f}")
plt.tight_layout()
plt.show()

tabla_d = pd.DataFrame(filas)
tabla_d.sort_values("brecha", ascending=False)

In [ ]:
# Autocomprobación — Parte D
assert len(tabla_d) == 9, "Deben ser 9 combinaciones (3 valores de C x 3 de gamma)"
assert {"C", "gamma", "train", "test"} <= set(tabla_d.columns)
print("✔ Parte D: grilla de 9 modelos completa")

**Pregunta D1:** ¿qué combinación de (`C`, `gamma`) subajusta y cuál sobreajusta? Justifica con la forma de la frontera y con la brecha entre exactitud de entrenamiento y de prueba.

**Respuesta:** (escribe aquí)

---
## Parte E — Datos reales con escalado (20 min)

**Qué hacer**

1. Carga `load_breast_cancer`. Separa entrenamiento/prueba (25 %, estratificado, `random_state=SEED`). **No uses el conjunto de prueba hasta la Parte F.**
2. Entrena `SVC(kernel="rbf")` **sin escalar** y luego dentro de `make_pipeline(StandardScaler(), SVC(kernel="rbf"))`.
3. Compara con validación cruzada estratificada de 5 particiones sobre el conjunto de entrenamiento (media ± desviación).
4. Mira qué variables tienen mayor y menor rango.

In [ ]:
datos = load_breast_cancer(as_frame=True)
X, y = datos.data, datos.target
# TODO: separa entrenamiento/prueba (test_size=0.25, stratify=y, random_state=SEED)
X_tr, X_te, y_tr, y_te = ...
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

sin_escalar = SVC(kernel="rbf")
# TODO: arma un Pipeline con StandardScaler() y SVC(kernel="rbf")
con_escalado = ...

# TODO: puntajes de validación cruzada (cross_val_score) de cada modelo sobre (X_tr, y_tr)
puntajes_sin = ...
puntajes_con = ...

print(f"Sin escalar : {puntajes_sin.mean():.3f} ± {puntajes_sin.std():.3f}")
print(f"Con escalado: {puntajes_con.mean():.3f} ± {puntajes_con.std():.3f}")

# TODO: rango (máx - mín) de cada variable en X_tr; muestra las 3 de mayor y las 3 de menor rango
...

In [ ]:
# Autocomprobación — Parte E
assert X_tr.shape[0] + X_te.shape[0] == len(X), "La partición no cubre todos los datos"
assert abs(len(X_te) / len(X) - 0.25) < 0.01, "El conjunto de prueba debe ser ~25 %"
assert len(puntajes_sin) == 5 and len(puntajes_con) == 5, "Usa 5 particiones"
assert hasattr(con_escalado, "steps"), "con_escalado debe ser un Pipeline"
print("✔ Parte E: comparación con y sin escalado lista")

**Pregunta E1:** ¿por qué empeora la SVM sin escalar? ¿Qué variables tienen mayor rango y cómo afectan la distancia que usa el kernel RBF?

**Respuesta:** (escribe aquí)

---
## Parte F — Afinar `C` y `gamma` (10 min)

**Qué hacer**

1. `GridSearchCV` sobre el Pipeline anterior con `C ∈ {0.1, 1, 10, 100}` y `gamma ∈ {1e-3, 1e-2, 1e-1, 1}`, con la misma validación cruzada (`cv`) y `scoring="accuracy"`. Recuerda que los hiperparámetros de un Pipeline se nombran `paso__parametro` (por ejemplo `svc__C`).
2. Reporta los mejores parámetros y su puntaje de validación.
3. Evalúa **una sola vez** en el conjunto de prueba.

> Esto anticipa la clase 9 (Grid y Random Search); hoy no se profundiza.

In [ ]:
pipe = con_escalado    # el mismo Pipeline de la Parte E (StandardScaler + SVC RBF)
# TODO: define la grilla con C ∈ {0.1, 1, 10, 100} y gamma ∈ {1e-3, 1e-2, 1e-1, 1}
# (pista: los nombres son "svc__C" y "svc__gamma")
grilla = {...}

# TODO: GridSearchCV(pipe, grilla, cv=cv, scoring="accuracy") y ajústalo con (X_tr, y_tr)
busqueda = ...

# TODO: imprime los mejores parámetros, el mejor puntaje de validación (best_score_)
# y la exactitud en el conjunto de prueba (una sola vez)
...

In [ ]:
# Autocomprobación — Parte F
assert hasattr(busqueda, "best_params_"), "Ajusta la búsqueda con .fit antes de continuar"
assert set(busqueda.best_params_) == {"svc__C", "svc__gamma"}, "Revisa los nombres de los hiperparámetros"
assert len(busqueda.cv_results_["params"]) == 16, "La grilla debe tener 4 x 4 = 16 combinaciones"
print("✔ Parte F: búsqueda completa")

**Pregunta F1:** ¿por qué se ajusta la búsqueda sólo con `X_tr` y el conjunto de prueba se evalúa una sola vez al final? ¿Qué pasaría si se eligiera `C` y `gamma` mirando el test?

**Respuesta:** (escribe aquí)

---
## Parte G — Comparación y cierre (5 min)

**Qué hacer**

1. Compara, con la **misma** validación cruzada `cv` sobre `(X_tr, y_tr)`, estos tres modelos:
   - `LogisticRegression(max_iter=1000)` dentro de un Pipeline con `StandardScaler`.
   - `RandomForestClassifier(n_estimators=200, random_state=SEED)`.
   - La búsqueda completa de la Parte F (`busqueda`, el objeto `GridSearchCV`): así `C` y `gamma` se vuelven a elegir dentro de cada partición (**validación cruzada anidada**).
2. Arma una tabla con media ± desviación de la exactitud de cada uno (`tabla_g`).
3. Responde la pregunta final.

> **¿Por qué no usar `busqueda.best_estimator_`?** Sus hiperparámetros se eligieron mirando estas mismas particiones: evaluarlo con ellas reproduce `best_score_`, que es optimista. Pasar el `GridSearchCV` completo a `cross_val_score` repite la búsqueda dentro de cada partición y permite comparar en igualdad de condiciones con los otros dos modelos, que no se afinaron.

In [ ]:
modelos = {
    # TODO: Pipeline(StandardScaler, LogisticRegression(max_iter=1000))
    "Regresión logística": ...,
    # TODO: RandomForestClassifier(n_estimators=200, random_state=SEED)
    "Random Forest": ...,
    # TODO: la búsqueda completa de la Parte F (pista: el objeto GridSearchCV `busqueda`)
    "SVM RBF (afinada)": ...,
}

filas = []
for nombre, modelo in modelos.items():
    # TODO: puntajes de validación cruzada con la MISMA `cv` sobre (X_tr, y_tr)
    p = ...
    filas.append({"modelo": nombre, "media": round(p.mean(), 3), "desviación": round(p.std(), 3),
                  "resumen": f"{p.mean():.3f} ± {p.std():.3f}"})

tabla_g = pd.DataFrame(filas).set_index("modelo")
tabla_g

In [ ]:
# Autocomprobación — Parte G
assert list(tabla_g.index) == ["Regresión logística", "Random Forest", "SVM RBF (afinada)"]
assert tabla_g["media"].between(0.8, 1.0).all(), "Las exactitudes deberían estar entre 0.8 y 1.0 en este dataset"
print("✔ Parte G: comparación lista")

**Pregunta final:** ¿cuándo elegirías SVM en vez de regresión logística o Random Forest? Responde en 3–4 líneas, apoyándote en la tabla y en lo visto en las partes A–F.

**Respuesta:** (escribe aquí)

---
## Entregable

- Este notebook completo, con las partes **A–G** ejecutadas (todas las celdas con salida visible) y las **respuestas escritas** en las celdas «Respuesta».
- La respuesta final de **3–4 líneas** a «¿cuándo elegirías SVM en vez de regresión logística o Random Forest?».
- **Entrega por AULA-USM.**

> Recordatorio: el **Certamen 2 es el miércoles 28 de octubre** (sesión completa) y cubre las clases 6, 7 y 8.